In [28]:
import requests
from bs4 import BeautifulSoup
import json
import time
from tqdm import tqdm

def clean(s):
    return "".join(c for c in s if not c.isdigit())

def load_dois(path):
    with open(path, "r") as f:
        dois = [line.strip() for line in f if line.strip()]

    return [d.replace("https://doi.org/", "") for d in dois]


def get_html(doi):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    try:
        # arXiv case
        if "arXiv" in doi:
            arxiv_id = doi.split("arXiv.")[-1]
            url = f"https://arxiv.org/html/{arxiv_id}"

            r = requests.get(url, headers=headers, timeout=15)

            if r.status_code == 200:
                return r.text
            else:
                print(f"[arXiv] {doi} - HTML not available")
                return None

        # normal DOI
        url = f"https://doi.org/{doi}"
        r = requests.get(url, headers=headers, allow_redirects=True, timeout=15)

        if r.status_code != 200:
            print(f"[DOI] {doi} - request failed ({r.status_code})")
            return None

        # heurystyka: paywall / login
        if any(x in r.text.lower() for x in [
            "access denied",
            "purchase",
            "buy article",
            "login",
            "sign in",
            "institutional access"
        ]):
            print(f"[DOI] {doi} - paywall or login required")
            return None
        return r.text

    except Exception:
        print(f"[DOI] {doi} - connection error")
        return None


def parse_html(html):
    soup = BeautifulSoup(html, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    abstract = ""
    meta = soup.find("meta", {"name": "description"})
    if meta:
        abstract = meta.get("content", "")

    sections = []

    headers = soup.find_all(["h1", "h2", "h3"])

    for tag in headers:
        heading = clean(tag.get_text(strip=True))

        if not heading or len(heading) < 3:
            continue

        if any(x in heading.lower() for x in [
            "reference", "supplement", "acknowledg",
            "download", "citation"
        ]):
            continue

        text_parts = []

        for sib in tag.find_next_siblings():
            if sib.name in ["h1", "h2", "h3"]:
                break

            txt = sib.get_text(" ", strip=True)
            if txt:
                text_parts.append(txt)

        section_text = " ".join(text_parts)

        if len(section_text) < 50:
            continue

        sections.append({
            "heading": heading,
            "text": section_text
        })

    return title, abstract, sections


def process_dois(dois, delay=1):
    results = []

    for doi in tqdm(dois, desc="Processing"):
        html = get_html(doi)

        if not html:
            continue

        title, abstract, sections = parse_html(html)

        if not title and not sections:
            print(f"[DOI] {doi} - no usable content")
            continue

        results.append({
            "paper_id": doi,
            "source": "HTML",
            "title": title,
            "abstract": abstract,
            "sections": sections
        })

        time.sleep(delay)

    return results


def save_json(data, path="articles.json"):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


if __name__ == "__main__":
    dois = load_dois("/home/libelt/Downloads/arxiv_test.txt")

    data = process_dois(dois, delay=1)

    print(f"\nCollected {len(data)} articles")

    #save_json(data)

Processing: 100%|█████████████████████████████████| 1/1 [00:01<00:00,  1.29s/it]


Collected 1 articles


In [22]:
# TODO:
# - Puścić na całym systematic_reviews_allergy, zobaczyć czy działa
# - Zrobić preprocessing dla htmla (żeby nie zapisywało 1. Introduction tylko Introduction)
# - NLP dla wydzielania sekcji
# - Histogram dla sekcji
# - Stworzyć DUŻY plik z doi zarówno z pubmeda jak i arxiva i puścić na nim final pipeline


In [29]:
def extract_section(data, section):
    for item in data:
        if item["heading"].casefold() == section.casefold():
            return item["text"]

In [30]:
def list_with_sections(data, section):
    result = []
    for item in data:
        items = []
        items.append(item['paper_id'])
        items.append(extract_section(item["sections"], section))
        result.append(items)
    return result

In [31]:
list_with_sections(data, "introduction")

[['10.48550/arXiv.2603.24132',
  'Conversational artificial intelligence has recently demonstrated strong potential for assisting users in healthcare settings, particularly for preliminary symptom assessment and medical guidance. Large language models (LLMs) have shown impressive capabilities in natural language understanding and dialogue generation, enabling systems to interact with patients in a conversational manner (Tu et al. , 2024 ) . However, many existing models primarily operate in a single-turn question–answering paradigm, where users provide all relevant information in a single prompt. In real clinical practice, physicians rarely rely on such interactions; instead, diagnosis typically emerges through a sequence of questions that progressively refine the patient’s symptoms. Furthermore, most conversational medical AI systems are trained on datasets that are either template-based or limited to a single language. While datasets such as MDDial (Macherla et al. , 2023 ) provide a